In [1]:
import polars as pl
from pathlib import Path
from datetime import timedelta

In [2]:
GTFS = Path("raw/processed_gtfs")
RT = Path("raw")
 
# ---------------------------------------------------
# Read files
# ---------------------------------------------------
 
routes = pl.read_parquet(GTFS / "routes.parquet")
trips = pl.read_parquet(GTFS / "trips.parquet")
stops = pl.read_parquet(GTFS / "stops.parquet")
stop_times = pl.read_parquet(GTFS / "stop_times.parquet")
 
trip_updates = pl.read_parquet(RT / "trip_updates/2026-07-01.parquet")
vehicle_positions = pl.read_parquet(RT / "vehicle_positions/2026-07-01.parquet")
 
print("Loaded")

Loaded


In [3]:
def gtfs_to_seconds(col):
    p = pl.col(col).str.split(":")
    return (
        p.list.get(0).cast(pl.Int32) * 3600
        + p.list.get(1).cast(pl.Int32) * 60
        + p.list.get(2).cast(pl.Int32)
    )
 
stop_times = stop_times.with_columns(
    pl.col("stop_sequence").cast(pl.UInt32),
    pl.col("stop_id").cast(pl.Utf8),
    gtfs_to_seconds("arrival_time").alias("scheduled_arrival"),
    gtfs_to_seconds("departure_time").alias("scheduled_departure"),
)
 
stops = stops.with_columns([
    pl.col("stop_id").cast(pl.Utf8),

    pl.col("stop_lat")
      .str.strip_chars()
      .cast(pl.Float64),

    pl.col("stop_lon")
      .str.strip_chars()
      .cast(pl.Float64),
])

In [4]:

trip_updates = (
    trip_updates
    .sort("feed_timestamp")
    .group_by(["trip_id", "start_date", "stop_sequence"])
    .last()
)

In [5]:
if "schedule_relationship" in trip_updates.columns:
    trip_updates = trip_updates.filter(pl.col("schedule_relationship") == 0)
 

In [6]:
trip_updates = trip_updates.with_columns(
    pl.from_epoch("arrival_time", time_unit="s").alias("event_time")
)
 
vehicle_positions = vehicle_positions.with_columns(
    pl.from_epoch("timestamp", time_unit="s").alias("vehicle_time")
)

In [7]:
TRIP_ID_PATTERN = r'^[A-Z]{2}_([A-Z0-9]+)-(\w+?)-(\d+)_([A-Z0-9+]+)_(\d+)$'
 
def parse_trip_id(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 2).alias("_service_day"),
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 3).alias("_origin_secs"),
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 5).alias("_trip_num"),
    ])
 
trips_parsed = parse_trip_id(trips)
tu_parsed = parse_trip_id(trip_updates)
 
# 1) direct trip_id match
direct = tu_parsed.join(
    trips_parsed.select(["trip_id", "route_id", "direction_id", "shape_id", "service_id"]),
    on="trip_id", how="inner", suffix="_static",
)

In [8]:
unmatched = tu_parsed.join(direct.select("trip_id").unique(), on="trip_id", how="anti")
fallback = unmatched.join(
    trips_parsed.select([
        "trip_id", "route_id", "direction_id", "shape_id", "service_id",
        "_service_day", "_origin_secs", "_trip_num",
    ]).rename({"trip_id": "_static_trip_id"}),
    on=["route_id", "_service_day", "_origin_secs", "_trip_num"],
    how="inner",
    suffix="_static",
).with_columns(
    pl.col("_static_trip_id").alias("trip_id")   # use the STATIC trip_id downstream
).drop("_static_trip_id")
 
n_direct, n_fallback, n_total = direct.height, fallback.height, tu_parsed.height
print(f"[join] direct match: {n_direct:,} | fallback match: {n_fallback:,} | "
      f"total: {(n_direct + n_fallback) / n_total:.1%} of {n_total:,} rows")
 
direct = direct.drop(["_service_day", "_origin_secs", "_trip_num"])
fallback = fallback.drop(["_service_day", "_origin_secs", "_trip_num"])

[join] direct match: 70,053 | fallback match: 0 | total: 100.0% of 70,053 rows


In [9]:
direct = (
    direct.drop(["route_id", "direction_id"])
          .rename({"route_id_static": "route_id", "direction_id_static": "direction_id"})
)
fallback = fallback.drop("direction_id").rename({"direction_id_static": "direction_id"})
 
matched = pl.concat([direct, fallback.select(direct.columns)])

In [10]:
data = (
    matched
    .join(
        stop_times.select([
            "trip_id", "stop_sequence", "stop_id",
            "scheduled_arrival", "scheduled_departure",
        ]),
        on=["trip_id", "stop_sequence"],
        how="inner",   # inner now: every row here already has a resolved static trip_id,
                        # so a missing stop_times row means bad data, not an expected gap
    )
    .join(
        stops.select(["stop_id", "stop_lat", "stop_lon"]),
        on="stop_id",
        how="left",
    )
)
 
print(data.shape)

(70053, 22)


In [11]:
vp = vehicle_positions.select([
    "vehicle_id", "vehicle_time", "latitude", "longitude", "bearing",
])
 
data = data.sort(["vehicle_id", "event_time"])
vp = vp.sort(["vehicle_id", "vehicle_time"])
 
data = data.join_asof(
    vp,
    left_on="event_time",
    right_on="vehicle_time",
    by="vehicle_id",
    strategy="backward",
    tolerance=timedelta(minutes=2),
)

C:\Users\ishan\AppData\Local\Temp\ipykernel_3344\1183410916.py:8: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  data = data.join_asof(


In [12]:
data = (
    data
    .sort(["trip_id", "stop_sequence"])
    .with_columns(
        pl.col("arrival_time").shift(-1).over("trip_id").alias("next_arrival_time")
    )
    .with_columns(
        (pl.col("next_arrival_time") - pl.col("arrival_time")).alias("travel_time")
    )
)
 
data = data.filter(
    (pl.col("travel_time") > 0) & (pl.col("travel_time") < 1800)   # 30 min cap
)

In [13]:
data = data.with_columns([
    pl.col("event_time").dt.hour().alias("hour"),
    pl.col("event_time").dt.weekday().alias("weekday"),
    pl.col("event_time").dt.month().alias("month"),
])
 
data = data.with_columns([
    (
        (pl.col("hour").is_between(7, 9)) |
        (pl.col("hour").is_between(16, 18))
    ).cast(pl.Int8).alias("is_peak")
])

In [14]:
data = data.with_columns([
    (
        pl.col("scheduled_arrival")
        - pl.col("scheduled_departure").shift(1).over("trip_id")
    ).alias("scheduled_segment_time"),
 
    (
        pl.col("stop_sequence") / pl.col("stop_sequence").max().over("trip_id")
    ).alias("trip_progress"),
])

In [15]:
features = [
    "route_id",
    "direction_id",
    "shape_id",
    "service_id",
 
    "stop_sequence",
    "trip_progress",
 
    "hour",
    "weekday",
    "month",
    "is_peak",
 
    "scheduled_arrival",
    "scheduled_departure",
    "scheduled_segment_time",
 
    "stop_lat",
    "stop_lon",
 
    "latitude",
    "longitude",
    "bearing",
]
target = "travel_time"
id_cols = ["trip_id", "start_date"]
 
print(data.columns)
 
# %%
model_ready = data.select(id_cols + features + [target]).drop_nulls(subset=features + [target])
 
print(model_ready.shape)
print(model_ready.null_count())
print(model_ready.head())

['trip_id', 'start_date', 'stop_sequence', 'feed_timestamp', 'fetch_timestamp', 'start_time', 'schedule_relationship', 'vehicle_id', 'trip_timestamp', 'stop_id', 'arrival_time', 'departure_time', 'event_time', 'route_id', 'direction_id', 'shape_id', 'service_id', 'stop_id_right', 'scheduled_arrival', 'scheduled_departure', 'stop_lat', 'stop_lon', 'vehicle_time', 'latitude', 'longitude', 'bearing', 'next_arrival_time', 'travel_time', 'hour', 'weekday', 'month', 'is_peak', 'scheduled_segment_time', 'trip_progress']
(59768, 21)
shape: (1, 21)
┌─────────┬────────────┬──────────┬─────────────┬───┬──────────┬───────────┬─────────┬─────────────┐
│ trip_id ┆ start_date ┆ route_id ┆ direction_i ┆ … ┆ latitude ┆ longitude ┆ bearing ┆ travel_time │
│ ---     ┆ ---        ┆ ---      ┆ d           ┆   ┆ ---      ┆ ---       ┆ ---     ┆ ---         │
│ u32     ┆ u32        ┆ u32      ┆ ---         ┆   ┆ u32      ┆ u32       ┆ u32     ┆ u32         │
│         ┆            ┆          ┆ u32         ┆ 

In [16]:
print(model_ready["travel_time"].describe())

shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 59768.0   │
│ null_count ┆ 0.0       │
│ mean       ┆ 98.16037  │
│ std        ┆ 102.53241 │
│ min        ┆ 1.0       │
│ 25%        ┆ 49.0      │
│ 50%        ┆ 80.0      │
│ 75%        ┆ 115.0     │
│ max        ┆ 1799.0    │
└────────────┴───────────┘


In [17]:
model_ready.group_by("route_id").len().sort("len", descending=True)

route_id,len
str,u32
"""M4""",14769
"""M101""",13013
"""M15""",11600
"""M1""",11288
"""M2""",9098


In [18]:
model_ready.select([
    pl.col("scheduled_arrival").min().alias("min"),
    pl.col("scheduled_arrival").max().alias("max"),
    pl.col("scheduled_arrival").mean().alias("mean"),
])

min,max,mean
i32,i32,f64
1241,95160,51872.899143


In [19]:
print(
    trip_updates.group_by("route_id").len().sort("len", descending=True)
)

print(
    matched.group_by("route_id").len().sort("len", descending=True)
)

print(
    model_ready.group_by("route_id").len().sort("len", descending=True)
)

shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 16698 │
│ M101     ┆ 15665 │
│ M15      ┆ 14044 │
│ M1       ┆ 13106 │
│ M2       ┆ 10540 │
└──────────┴───────┘
shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 16698 │
│ M101     ┆ 15665 │
│ M15      ┆ 14044 │
│ M1       ┆ 13106 │
│ M2       ┆ 10540 │
└──────────┴───────┘
shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 14769 │
│ M101     ┆ 13013 │
│ M15      ┆ 11600 │
│ M1       ┆ 11288 │
│ M2       ┆ 9098  │
└──────────┴───────┘


In [20]:
model_ready.write_parquet("raw/processed_gtfs/baseline_dataset.parquet")
print("written")

written
